# 📊 CSE573 Group 17 - RAG Chatbot Evaluation

## Movie Recommendation System: Post-Chat Evaluation

This notebook evaluates the RAG (Retrieval-Augmented Generation) chatbot component of our KGNN-based movie recommendation system.

### Evaluation Metrics (from Project Proposal)
| Metric | Target | Description |
|--------|--------|-------------|
| Answer Relevancy | ≥0.70 | Embedding similarity between query and response |
| Faithfulness | ≥0.75 | Facts grounded in retrieved data |
| Query Understanding | ≥0.80 | Correct parsing of constraints |
| Response Time | <3s | Latency per query |

### Team
- Mannan Anand (Evaluation Lead)
- Shreya Marria, Sanchit Gupta, Matthew Mulderink, Maharshi Saragadam


## 1. Setup & Installation


In [ ]:
!pip -q install pandas networkx matplotlib rapidfuzz \
  sentence-transformers faiss-cpu transformers accelerate seaborn scipy


In [ ]:
import os, re, json, time, warnings, unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass, field
import pandas as pd
import numpy as np
import networkx as nx
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from rapidfuzz import process, fuzz
import faiss

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150

print("✓ All imports successful")


## 2. Load Data & Build Knowledge Graph

Replicating the core components from the main KGNN-RAG notebook.


In [ ]:
# Load MovieLens data
DATA_DIR = "/content"  # Update if needed

users = pd.read_csv(os.path.join(DATA_DIR, "users.csv"))
movies = pd.read_csv(os.path.join(DATA_DIR, "movies.csv"))
ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"))

# Filter to good movies (avg rating > 4.0) for quality recommendations
min_avg_rating = 4.0
movie_avg_ratings = ratings.groupby('MovieID')['Rating'].mean()
good_movies = movie_avg_ratings[movie_avg_ratings > min_avg_rating].index
movies = movies[movies['MovieID'].isin(good_movies)]
ratings = ratings[ratings['MovieID'].isin(good_movies)]

print(f"✓ Loaded {len(users):,} users, {len(movies):,} movies, {len(ratings):,} ratings")


In [ ]:
# Build Knowledge Graph
G = nx.DiGraph()

occupation_map = {
    0: "other", 1: "academic/educator", 2: "artist", 3: "clerical/admin",
    4: "college/grad student", 5: "customer service", 6: "doctor/health care",
    7: "executive/managerial", 8: "farmer", 9: "homemaker", 10: "K-12 student",
    11: "lawyer", 12: "programmer", 13: "retired", 14: "sales/marketing",
    15: "scientist", 16: "self-employed", 17: "technician/engineer",
    18: "tradesman/craftsman", 19: "unemployed", 20: "writer"
}

# Add user nodes
for _, user in users.iterrows():
    user_id = f"user_{user['UserID']}"
    G.add_node(user_id, ntype="user", user_id=int(user['UserID']))
    
    gender_id = f"gender_{user['Gender']}"
    G.add_node(gender_id, ntype="gender", name=user['Gender'])
    G.add_edge(user_id, gender_id, etype="is_gender")
    
    age_id = f"age_{user['Age'].replace('-', '_').replace('+', 'plus').replace('<', 'under')}"
    G.add_node(age_id, ntype="age_bucket", name=user['Age'])
    G.add_edge(user_id, age_id, etype="age_bucket")
    
    occ_code = int(user['Occupation'])
    occ_id = f"occ_{occ_code}"
    G.add_node(occ_id, ntype="occupation", name=occupation_map.get(occ_code, "other"), label=occupation_map.get(occ_code, "other"))
    G.add_edge(user_id, occ_id, etype="has_occupation")

# Add movie nodes
for _, movie in movies.iterrows():
    movie_id = f"movie_{movie['MovieID']}"
    G.add_node(movie_id, ntype="movie", movie_id=int(movie['MovieID']),
               title=movie['Title'], year=int(movie['Year']) if pd.notna(movie['Year']) else None)
    
    if pd.notna(movie['Genres']):
        for genre in movie['Genres'].split('|'):
            genre = genre.strip()
            genre_id = f"genre_{genre.replace(' ', '_').replace('-', '_')}"
            G.add_node(genre_id, ntype="genre", name=genre, title=genre)
            G.add_edge(movie_id, genre_id, etype="is_genre")
    
    if pd.notna(movie['Year']):
        decade = (int(movie['Year']) // 10) * 10
        year_id = f"year_{decade}s"
        G.add_node(year_id, ntype="year", name=f"{decade}s", decade=decade)
        G.add_edge(movie_id, year_id, etype="year_bucket")

# Add rating edges
for _, rating in ratings.iterrows():
    user_id = f"user_{rating['UserID']}"
    movie_id = f"movie_{rating['MovieID']}"
    rating_val = float(rating['Rating'])
    
    if user_id in G and movie_id in G:
        if rating_val >= 4.0:
            G.add_edge(user_id, movie_id, etype="rated_high", rating=rating_val)
        elif rating_val <= 2.0:
            G.add_edge(user_id, movie_id, etype="rated_low", rating=rating_val)
        else:
            G.add_edge(user_id, movie_id, etype="rated_medium", rating=rating_val)

print(f"✓ Graph built: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")


## 3. Build RAG Components


In [ ]:
# Precompute stats and helper functions
avg_by_movie = ratings.groupby("MovieID")["Rating"].mean().to_dict()
count_by_movie = ratings.groupby("MovieID")["Rating"].size().to_dict()
_movie_stats = ratings.groupby('MovieID').Rating.agg(['count','mean']).rename(columns={'count':'num','mean':'avg'})
MOVIE_STATS = _movie_stats.to_dict(orient='index')

def facts_for_movie(node_id: str):
    mid = int(str(node_id).split('_', 1)[1])
    mrow = movies.loc[movies['MovieID'] == mid]
    if mrow.empty:
        return {'title':'?', 'year':None, 'decade':None, 'genres':[], 'avg':0.0, 'num':0}
    m = mrow.iloc[0]
    year = int(m['Year']) if pd.notna(m['Year']) else None
    decade = f"{(year//10)*10}s" if year else None
    genres = [g.strip() for g in str(m['Genres']).split('|')] if pd.notna(m['Genres']) else []
    st = MOVIE_STATS.get(mid, {'num':0, 'avg':0.0})
    return {'title': str(m['Title']), 'year': year, 'decade': decade, 'genres': genres,
            'avg': float(st['avg']) if st['num'] else 0.0, 'num': int(st['num']) if st['num'] else 0}

# Build document corpus for RAG
docs = []
for n, data in G.nodes(data=True):
    if data.get("ntype") == "movie":
        mid = data["movie_id"]
        title = data.get("title", f"movie {mid}")
        year = data.get("year")
        movie_genres = [G.nodes[g]["name"] for g in G.neighbors(n) if G.nodes[g].get("ntype") == "genre"]
        decade = None
        for y in G.neighbors(n):
            if G.nodes[y].get("ntype") == "year":
                decade = G.nodes[y].get("name")
        num_r = int(count_by_movie.get(mid, 0))
        avg_r = float(avg_by_movie.get(mid, 0.0))
        text = f"{title} ({year}). Genres: {', '.join(movie_genres)}. Decade: {decade}. Popularity: {num_r} ratings. Average rating: {avg_r:.2f}."
        docs.append({"id": n, "kind": "movie", "title": title, "text": text,
                     "meta": {"year": year, "genres": movie_genres, "decade": decade, "num_ratings": num_r, "avg_rating": avg_r}})

print(f"✓ Built {len(docs)} movie documents for RAG")


In [ ]:
# Load embedding model and build FAISS index
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
emb_model = SentenceTransformer(EMB_MODEL)

print("Encoding movie documents...")
embs = emb_model.encode([d["text"] for d in docs], show_progress_bar=True, normalize_embeddings=True).astype("float32")

dim = embs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embs)

def semantic_search(query: str, k: int = 25):
    q = emb_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q, k)
    return list(zip(idxs[0].tolist(), scores[0].tolist()))

row2doc = docs
print(f"✓ FAISS index built with {index.ntotal} vectors")


In [ ]:
# Item-Item Collaborative Filtering
_hi = ratings.loc[ratings["Rating"] >= 4, ["UserID","MovieID"]].copy()
_uids = np.sort(_hi["UserID"].unique())
_mids = np.sort(movies["MovieID"].unique())
_uid2i = {u:i for i,u in enumerate(_uids)}
_mid2i = {m:i for i,m in enumerate(_mids)}
_i2mid = np.array(_mids)

row = _hi["MovieID"].map(_mid2i).dropna().astype(int).values
col = _hi["UserID"].map(_uid2i).dropna().astype(int).values
min_len = min(len(row), len(col))
row, col = row[:min_len], col[:min_len]
data = np.ones(min_len, dtype=np.float32)

R = sp.csr_matrix((data, (row, col)), shape=(len(_mids), len(_uids))).tocsr()
row_sq_sum = np.asarray(R.power(2).sum(axis=1)).ravel()
row_norm = np.sqrt(np.maximum(row_sq_sum, 1.0))
R_norm = R.multiply(1.0 / row_norm[:, None]).tocsr()
R_bool = R.astype(bool).tocsr().astype(np.int8)

def cf_similar_movies(mid: int, topk: int = 200, min_overlap: int = 20):
    i = _mid2i.get(mid, None)
    if i is None: return []
    ri_norm = R_norm.getrow(i)
    sims = (ri_norm @ R_norm.T).toarray().ravel()
    sims[i] = 0.0
    ri_bool = R_bool.getrow(i)
    ov = (ri_bool @ R_bool.T).toarray().ravel()
    mask = (ov >= min_overlap) & (sims > 0)
    idxs = np.where(mask)[0]
    if idxs.size == 0: return []
    order = idxs[np.argsort(-sims[idxs])]
    return [(f"movie_{int(_i2mid[j])}", float(sims[j]), int(ov[j])) for j in order[:topk]]

print(f"✓ CF matrix built: {R.shape}")


In [ ]:
# Title matching helpers
ARTICLES = {"the","a","an"}

def _strip_year_suffix(s: str) -> str:
    return re.sub(r"\s*\(\d{4}\)\s*$", "", s.strip())

def _move_trailing_article(s: str) -> str:
    m = re.match(r"^(.*),\s*(The|A|An)$", s, flags=re.I)
    return f"{m.group(2)} {m.group(1)}" if m else s

def _normalize_tokens(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii","ignore").decode("ascii")
    s = s.lower()
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def _variants(raw_title: str, year):
    base = _strip_year_suffix(raw_title)
    with_art = _move_trailing_article(base)
    toks1 = _normalize_tokens(base)
    toks2 = _normalize_tokens(with_art)
    vs = {toks1, toks2}
    if year and str(year).isdigit():
        y = str(int(year))
        vs |= {f"{toks1} {y}", f"{toks2} {y}"}
    for t in list(vs):
        words = [w for w in t.split() if w not in ARTICLES]
        if words:
            vs.add(" ".join(words))
    return vs

TITLE_VAR_INDEX = {}
TITLE_KEYS = []
TITLE_DISPLAY = {}

for _, row in movies.iterrows():
    node_id = f"movie_{int(row['MovieID'])}"
    raw = str(row["Title"])
    yr = row.get("Year", None)
    for v in _variants(raw, yr):
        TITLE_VAR_INDEX.setdefault(v, (node_id, raw))
        TITLE_KEYS.append(v)
    TITLE_DISPLAY[node_id] = raw

print(f"✓ Title index built with {len(TITLE_VAR_INDEX)} variants")


In [ ]:
# Constraint extraction and title resolver
all_genres = {G.nodes[n]["name"].lower() for n, d in G.nodes(data=True) if d.get("ntype") == "genre"}

def extract_constraints(text: str):
    tg = set()
    tl = text.lower()
    for g in all_genres:
        if re.search(rf"\b{re.escape(g)}\b", tl):
            tg.add(g)
    decade = None
    m = re.search(r"(?:19|20)?\d0s", tl)
    if m:
        s = m.group(0)
        decade = f"19{s}" if re.fullmatch(r"\d0s", s) else s
    return {"genres": list(tg), "decade": decade}

_title_ref_re = re.compile(r"(?:like|similar to|if i (?:liked|love[dn]?))\s+\"?(.+?)\"?$", re.IGNORECASE)
ACCEPT_THRESH = 93
SUGGEST_THRESH = 86

def _canon_query(s: str) -> str:
    return _normalize_tokens(_move_trailing_article(_strip_year_suffix(s)))

def resolve_title_flexible(user_text: str):
    text = user_text.strip()
    m = _title_ref_re.search(text)
    if m:
        cand = m.group(1)
        qkey = _canon_query(cand)
        hit = TITLE_VAR_INDEX.get(qkey)
        if hit:
            return [hit[0]], None, None
        best = process.extractOne(qkey, TITLE_KEYS, scorer=fuzz.token_set_ratio)
        if not best:
            return [], cand, None
        key, score, _ = best
        node, disp = TITLE_VAR_INDEX[key]
        if score >= ACCEPT_THRESH:
            return [node], None, None
        elif score >= SUGGEST_THRESH:
            return [], cand, disp
        else:
            return [], cand, None
    
    words = text.split()
    if len(words) <= 6 and not any(kw in text.lower() for kw in ['movies', 'films', 'for', 'recommend']):
        qkey = _canon_query(text)
        hit = TITLE_VAR_INDEX.get(qkey)
        if hit:
            return [hit[0]], None, None
        best = process.extractOne(qkey, TITLE_KEYS, scorer=fuzz.token_set_ratio)
        if best:
            key, score, _ = best
            if score >= 95:
                node, disp = TITLE_VAR_INDEX[key]
                return [node], None, None
    return [], None, None

# Topic guards and helpers
def _genres_for(node_id):
    f = facts_for_movie(node_id)
    return set(f.get("genres") or [])

def _year_for(node_id):
    return facts_for_movie(node_id).get("year", None)

def _passes_topic_guard(seed_genres, cand_genres, seed_year, cand_year, require_genre_overlap=True, max_year_gap=20):
    if require_genre_overlap and seed_genres and cand_genres:
        if seed_genres.isdisjoint(cand_genres):
            return False
    if seed_year and cand_year:
        if abs(int(seed_year) - int(cand_year)) > max_year_gap:
            return False
    return True

print(f"✓ Found {len(all_genres)} genres: {sorted(all_genres)}")


In [ ]:
# Main recommendation function
def ask_structured(query: str, top_k: int = 5, forced_genres=None, forced_decade=None):
    cons = extract_constraints(query)
    if forced_genres:
        cons["genres"] = list({g.lower() for g in forced_genres})
    if forced_decade:
        cons["decade"] = forced_decade

    seeds_resolved, oov_title, suggestion = resolve_title_flexible(query)
    seed_nodes = seeds_resolved
    candidates = {}

    if seed_nodes:
        seed_genres_union = set()
        seed_years = []
        for sn in seed_nodes:
            seed_genres_union |= _genres_for(sn)
            y = _year_for(sn)
            if y: seed_years.append(int(y))
        seed_year = int(np.median(seed_years)) if seed_years else None

        for sn in seed_nodes:
            try:
                mid = int(str(sn).split("_", 1)[1])
            except:
                continue
            for node, cf_score, overlap in cf_similar_movies(mid, topk=400, min_overlap=15):
                if node in seed_nodes:
                    continue
                if not _passes_topic_guard(seed_genres_union, _genres_for(node), seed_year, _year_for(node)):
                    continue
                prev = candidates.get(node, {"cf":0.0, "sem":0.0, "ov":0})
                prev["cf"] = max(prev["cf"], cf_score)
                prev["ov"] = max(prev["ov"], overlap)
                candidates[node] = prev

        try:
            sem_hits = semantic_search(query, k=300)
            sem_map = {row2doc[i]["id"]: sc for i, sc in sem_hits}
            for node in candidates.keys():
                candidates[node]["sem"] = float(sem_map.get(node, 0.0))
        except:
            pass
    else:
        sem_hits = semantic_search(query, k=600)
        for i, sc in sem_hits:
            node = row2doc[i]["id"]
            candidates[node] = {"cf":0.0, "sem": float(sc), "ov":0}

    def _passes_explicit(node):
        f = facts_for_movie(node)
        if cons["genres"]:
            cand = {g.lower() for g in (f.get("genres") or [])}
            if cand.isdisjoint(set(cons["genres"])): return False
        if cons["decade"]:
            if (f.get("decade") or "").lower() != cons["decade"].lower(): return False
        return True

    filtered = [n for n in candidates.keys()] if not (cons["genres"] or cons["decade"]) else \
               [n for n in candidates.keys() if _passes_explicit(n)]

    scored = []
    for n in filtered:
        f = facts_for_movie(n)
        pop = np.log1p(f.get("num", 0)) / 10.0
        cf = candidates[n]["cf"]
        sem = candidates[n]["sem"]
        score = (0.9*cf + 0.08*sem + 0.02*pop) if seed_nodes else (0.85*sem + 0.15*pop)
        scored.append((n, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    top = scored[:top_k]

    out = []
    for n, _ in top:
        f = facts_for_movie(n)
        out.append({"title": f["title"], "year": f.get("year","?"), "genres": f.get("genres", []),
                    "avg": round(f.get("avg", 0.0), 2), "num": f.get("num", 0), "decade": f.get("decade")})
    return out

print("✓ Recommendation engine ready")


## 4. Define Evaluation Test Suite

Comprehensive test cases covering different query types.


In [ ]:
# Test cases with ground truth
TEST_QUERIES = [
    # Title-based similarity queries
    {"query": "Movies like The Matrix", "type": "title_similarity",
     "expected_genres": ["sci-fi", "action"], "seed_title": "Matrix"},
    {"query": "Similar to Toy Story", "type": "title_similarity",
     "expected_genres": ["animation", "comedy", "children's"], "seed_title": "Toy Story"},
    {"query": "If I liked Schindler's List", "type": "title_similarity",
     "expected_genres": ["drama", "war"], "seed_title": "Schindler's List"},
    
    # Genre-based queries
    {"query": "Recommend me some comedy movies", "type": "genre",
     "expected_genres": ["comedy"], "constraints": {"genre": "comedy"}},
    {"query": "I want to watch horror films", "type": "genre",
     "expected_genres": ["horror"], "constraints": {"genre": "horror"}},
    {"query": "Action adventure movies please", "type": "genre",
     "expected_genres": ["action", "adventure"], "constraints": {"genre": "action"}},
    {"query": "Good drama films", "type": "genre",
     "expected_genres": ["drama"], "constraints": {"genre": "drama"}},
    
    # Decade/temporal queries
    {"query": "Classic movies from the 90s", "type": "decade",
     "expected_decade": "1990s", "constraints": {"decade": "1990s"}},
    {"query": "80s action films", "type": "decade_genre",
     "expected_decade": "1980s", "expected_genres": ["action"]},
    {"query": "Movies from 2000s", "type": "decade",
     "expected_decade": "2000s", "constraints": {"decade": "2000s"}},
    
    # Combined queries
    {"query": "Sci-fi movies from the 1990s like Star Wars", "type": "combined",
     "expected_genres": ["sci-fi"], "expected_decade": "1990s"},
    {"query": "90s comedy similar to Home Alone", "type": "combined",
     "expected_genres": ["comedy"], "expected_decade": "1990s"},
    
    # Complex natural language
    {"query": "I'm looking for feel-good family movies", "type": "semantic",
     "expected_genres": ["comedy", "family", "children's"]},
    {"query": "Thrilling suspense movies that keep you on edge", "type": "semantic",
     "expected_genres": ["thriller", "mystery"]},
    {"query": "Epic war movies with great storytelling", "type": "semantic",
     "expected_genres": ["war", "drama"]},
    
    # Edge cases
    {"query": "The Godfather", "type": "bare_title", "seed_title": "Godfather"},
    {"query": "Best movies ever", "type": "vague"},
    {"query": "Movies like Avatar 2", "type": "oov_title", "expected_oov": True},
]

print(f"✓ Defined {len(TEST_QUERIES)} test queries")


## 5. Run Evaluation

Execute test queries and collect metrics.


In [ ]:
# Evaluation metrics storage
results = {
    "query": [], "type": [], "response_time_ms": [], "num_recommendations": [],
    "genre_match_score": [], "decade_match": [], "title_resolved": [],
    "avg_movie_rating": [], "avg_popularity": [], "query_relevancy_score": [],
}

print("Running evaluation...")
print("="*80)

for i, test in enumerate(TEST_QUERIES):
    query = test["query"]
    qtype = test["type"]
    
    # Measure response time
    start_time = time.time()
    recs = ask_structured(query, top_k=5)
    response_time = (time.time() - start_time) * 1000
    
    # Check title resolution
    seeds, oov, suggestion = resolve_title_flexible(query)
    title_resolved = len(seeds) > 0
    
    # Calculate genre match score
    genre_match = 0.0
    if "expected_genres" in test and recs:
        expected = set(g.lower() for g in test["expected_genres"])
        matches = sum(1 for rec in recs if set(g.lower() for g in rec.get("genres", [])) & expected)
        genre_match = matches / len(recs) if recs else 0
    
    # Check decade match
    decade_match = 0
    if "expected_decade" in test and recs:
        expected_decade = test["expected_decade"].lower()
        matches = sum(1 for r in recs if r.get("decade", "").lower() == expected_decade)
        decade_match = matches / len(recs) if recs else 0
    
    # Calculate query-response relevancy using embeddings
    if recs:
        rec_texts = [f"{r['title']} ({r['year']}) - {', '.join(r['genres'])}" for r in recs]
        query_emb = emb_model.encode([query], normalize_embeddings=True)
        rec_embs = emb_model.encode(rec_texts, normalize_embeddings=True)
        relevancy_scores = np.dot(rec_embs, query_emb.T).flatten()
        avg_relevancy = float(np.mean(relevancy_scores))
    else:
        avg_relevancy = 0.0
    
    # Store results
    results["query"].append(query)
    results["type"].append(qtype)
    results["response_time_ms"].append(response_time)
    results["num_recommendations"].append(len(recs))
    results["genre_match_score"].append(genre_match)
    results["decade_match"].append(decade_match)
    results["title_resolved"].append(title_resolved)
    results["avg_movie_rating"].append(np.mean([r["avg"] for r in recs]) if recs else 0)
    results["avg_popularity"].append(np.mean([r["num"] for r in recs]) if recs else 0)
    results["query_relevancy_score"].append(avg_relevancy)
    
    print(f"\n[{i+1}/{len(TEST_QUERIES)}] {qtype.upper()}: \"{query}\"")
    print(f"  ⏱ Response time: {response_time:.1f}ms | 📊 Recs: {len(recs)} | 🎯 Genre: {genre_match:.2f} | 🔗 Relevancy: {avg_relevancy:.3f}")
    if recs:
        print(f"  Top 3: {[r['title'] for r in recs[:3]]}")

print("\n" + "="*80)
print("✓ Evaluation complete")


## 6. Calculate Aggregate Metrics & Faithfulness


In [ ]:
# Convert to DataFrame and calculate aggregate metrics
results_df = pd.DataFrame(results)

print("="*80)
print("AGGREGATE EVALUATION METRICS")
print("="*80)

metrics = {
    "Avg Response Time (ms)": results_df["response_time_ms"].mean(),
    "Avg Recommendations": results_df["num_recommendations"].mean(),
    "Genre Match Rate": results_df["genre_match_score"].mean(),
    "Avg Answer Relevancy": results_df["query_relevancy_score"].mean(),
    "Avg Movie Rating": results_df["avg_movie_rating"].mean(),
}

targets = {"Avg Response Time (ms)": 3000, "Avg Answer Relevancy": 0.70, "Genre Match Rate": 0.60}

for metric, value in metrics.items():
    target = targets.get(metric)
    if target:
        if metric == "Avg Response Time (ms)":
            status = "✓ PASS" if value < target else "✗ FAIL"
        else:
            status = "✓ PASS" if value >= target else "✗ FAIL"
        print(f"  {metric}: {value:.3f} (Target: {target}) {status}")
    else:
        print(f"  {metric}: {value:.3f}")

# Faithfulness check
print("\n" + "-"*80)
print("FAITHFULNESS EVALUATION")
print("-"*80)

all_movie_titles = set(movies["Title"].str.lower())
faith_tests = [
    {"query": "Movies like The Matrix", "type": "similarity"},
    {"query": "Best comedy films", "type": "genre"},
    {"query": "90s action movies", "type": "decade"},
]

faith_scores = []
for test in faith_tests:
    recs = ask_structured(test["query"], top_k=10)
    grounded = sum(1 for rec in recs if rec["title"].lower() in all_movie_titles)
    score = grounded / len(recs) if recs else 0
    faith_scores.append(score)
    print(f"  {test['type'].upper()}: {grounded}/{len(recs)} grounded ({score:.1%})")

avg_faithfulness = np.mean(faith_scores)
print(f"\nOVERALL FAITHFULNESS: {avg_faithfulness:.3f} (Target: ≥0.75) {'✓ PASS' if avg_faithfulness >= 0.75 else '✗ FAIL'}")


In [ ]:
# Query Understanding Evaluation
print("="*80)
print("QUERY UNDERSTANDING EVALUATION")
print("="*80)

constraint_tests = [
    {"query": "comedy movies from the 90s", "expected": {"genres": ["comedy"], "decade": "1990s"}},
    {"query": "action adventure films", "expected": {"genres": ["action", "adventure"], "decade": None}},
    {"query": "80s sci-fi", "expected": {"genres": ["sci-fi"], "decade": "1980s"}},
    {"query": "drama films from 2000s", "expected": {"genres": ["drama"], "decade": "2000s"}},
    {"query": "mystery thriller", "expected": {"genres": ["mystery", "thriller"], "decade": None}},
]

constraint_scores = []
for test in constraint_tests:
    parsed = extract_constraints(test["query"])
    expected = test["expected"]
    
    expected_genres = set(g.lower() for g in expected.get("genres", []))
    parsed_genres = set(g.lower() for g in parsed.get("genres", []))
    genre_match = len(expected_genres & parsed_genres) / len(expected_genres) if expected_genres else 1.0
    decade_match = 1.0 if parsed.get("decade") == expected.get("decade") else 0.0
    
    score = (genre_match + decade_match) / 2
    constraint_scores.append(score)
    
    status = "✓" if score >= 0.8 else "✗"
    print(f"\n{status} \"{test['query']}\"")
    print(f"  Expected: {expected} | Parsed: {parsed} | Score: {score:.2f}")

avg_constraint_score = np.mean(constraint_scores)
print(f"\n{'='*80}")
print(f"QUERY UNDERSTANDING: {avg_constraint_score:.3f} (Target: ≥0.80) {'✓ PASS' if avg_constraint_score >= 0.80 else '✗ FAIL'}")


## 7. Visualizations


In [ ]:
# Create evaluation visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Response Time Distribution
ax1 = axes[0, 0]
ax1.hist(results_df["response_time_ms"], bins=15, color='steelblue', alpha=0.7, edgecolor='black')
ax1.axvline(x=3000, color='red', linestyle='--', label='Target (<3s)')
ax1.axvline(x=results_df["response_time_ms"].mean(), color='green', linestyle='-', label=f'Mean ({results_df["response_time_ms"].mean():.0f}ms)')
ax1.set_xlabel('Response Time (ms)')
ax1.set_ylabel('Count')
ax1.set_title('Response Time Distribution', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Relevancy Score Distribution
ax2 = axes[0, 1]
ax2.hist(results_df["query_relevancy_score"], bins=15, color='mediumseagreen', alpha=0.7, edgecolor='black')
ax2.axvline(x=0.70, color='red', linestyle='--', label='Target (≥0.70)')
ax2.axvline(x=results_df["query_relevancy_score"].mean(), color='green', linestyle='-', label=f'Mean ({results_df["query_relevancy_score"].mean():.3f})')
ax2.set_xlabel('Relevancy Score')
ax2.set_ylabel('Count')
ax2.set_title('Query-Response Relevancy', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Metrics by Query Type
ax3 = axes[1, 0]
type_relevancy = results_df.groupby("type")["query_relevancy_score"].mean().sort_values(ascending=True)
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(type_relevancy)))
bars = ax3.barh(range(len(type_relevancy)), type_relevancy.values, color=colors)
ax3.set_yticks(range(len(type_relevancy)))
ax3.set_yticklabels(type_relevancy.index)
ax3.axvline(x=0.70, color='red', linestyle='--', label='Target')
ax3.set_xlabel('Avg Relevancy Score')
ax3.set_title('Relevancy by Query Type', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='x')

# 4. Summary Metrics
ax4 = axes[1, 1]
summary_metrics = {
    'Response\nTime': min(results_df["response_time_ms"].mean() / 3000, 1.0),
    'Answer\nRelevancy': results_df["query_relevancy_score"].mean(),
    'Genre\nMatch': results_df["genre_match_score"].mean(),
    'Faithfulness': avg_faithfulness,
    'Query\nUnderstanding': avg_constraint_score,
}
targets_norm = {'Response\nTime': 1.0, 'Answer\nRelevancy': 0.70, 'Genre\nMatch': 0.60, 'Faithfulness': 0.75, 'Query\nUnderstanding': 0.80}

x = np.arange(len(summary_metrics))
width = 0.35
bars1 = ax4.bar(x - width/2, list(summary_metrics.values()), width, label='Actual', color='steelblue', alpha=0.8)
bars2 = ax4.bar(x + width/2, list(targets_norm.values()), width, label='Target', color='coral', alpha=0.8)
ax4.set_ylabel('Score')
ax4.set_title('Summary Metrics vs Targets', fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(list(summary_metrics.keys()), fontsize=9)
ax4.legend()
ax4.set_ylim(0, 1.1)
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('chatbot_evaluation_metrics.png', dpi=300, bbox_inches='tight')
print("✓ Saved: chatbot_evaluation_metrics.png")
plt.show()


## 8. Final Summary Report


In [ ]:
# Generate and save summary report
summary_data = {
    'Metric': ['Avg Response Time (ms)', 'Answer Relevancy', 'Faithfulness', 'Query Understanding', 'Genre Match Rate', 'Avg Movie Rating'],
    'Actual': [f"{results_df['response_time_ms'].mean():.1f}", f"{results_df['query_relevancy_score'].mean():.3f}",
               f"{avg_faithfulness:.3f}", f"{avg_constraint_score:.3f}", f"{results_df['genre_match_score'].mean():.3f}",
               f"{results_df['avg_movie_rating'].mean():.2f}"],
    'Target': ['<3000', '≥0.70', '≥0.75', '≥0.80', '≥0.60', 'N/A'],
    'Status': ['✓' if results_df['response_time_ms'].mean() < 3000 else '✗',
               '✓' if results_df['query_relevancy_score'].mean() >= 0.70 else '✗',
               '✓' if avg_faithfulness >= 0.75 else '✗', '✓' if avg_constraint_score >= 0.80 else '✗',
               '✓' if results_df['genre_match_score'].mean() >= 0.60 else '✗', '—']
}

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv('chatbot_evaluation_summary.csv', index=False)
results_df.to_csv('chatbot_evaluation_detailed.csv', index=False)

print("="*80)
print("CSE573 GROUP 17 - RAG CHATBOT EVALUATION REPORT")
print("="*80)
print("\n📊 RESULTS SUMMARY")
print("-"*80)
print(summary_df.to_string(index=False))

passes = sum(1 for s in summary_data['Status'] if s == '✓')
total = sum(1 for s in summary_data['Status'] if s in ['✓', '✗'])
print(f"\n🎯 TARGETS MET: {passes}/{total} ({passes/total*100:.1f}%)")

print("\n📈 KEY FINDINGS")
print("-"*80)
print(f"• Response time: {results_df['response_time_ms'].mean():.1f}ms (well under 3s)")
print(f"• Relevancy: {results_df['query_relevancy_score'].mean():.3f} (semantic alignment)")
print(f"• Faithfulness: {avg_faithfulness:.3f} (recommendations grounded in KG)")
print(f"• Query parsing: {avg_constraint_score:.3f}")

print("\n📁 FILES GENERATED")
print("-"*80)
print("• chatbot_evaluation_metrics.png")
print("• chatbot_evaluation_summary.csv")
print("• chatbot_evaluation_detailed.csv")

print("\n" + "="*80)
print("EVALUATION COMPLETE")
print("="*80)


## 9. Comparison with Pre-Chat Evaluation

Comparing RAG chatbot performance with the underlying KGNN recommendation evaluation.


In [ ]:
print("="*80)
print("COMPARISON: PRE-CHAT (KGNN) vs POST-CHAT (RAG)")
print("="*80)

# Pre-chat results from your evaluation_summary.csv
pre_chat = {
    "Precision@1": 0.3296, "Precision@5": 0.2373, "AUC-PR": 0.8368,
    "Binary Precision": 0.9055, "Recall": 0.0344, "Targets Met": "2/9 (22%)"
}

# Post-chat (RAG) results
post_chat = {
    "Answer Relevancy": results_df['query_relevancy_score'].mean(),
    "Faithfulness": avg_faithfulness,
    "Query Understanding": avg_constraint_score,
    "Response Time": f"{results_df['response_time_ms'].mean():.0f}ms",
    "Targets Met": f"{passes}/{total} ({passes/total*100:.0f}%)"
}

print("\n📊 PRE-CHAT (KGNN Recommendation Quality)")
print("-"*40)
for k, v in pre_chat.items():
    print(f"  {k}: {v}")

print("\n💬 POST-CHAT (RAG Chatbot Quality)")
print("-"*40)
for k, v in post_chat.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.3f}")
    else:
        print(f"  {k}: {v}")

print("""
📝 INTERPRETATION
-----------------
The RAG chatbot evaluation complements the pre-chat KGNN evaluation:

✅ PRE-CHAT STRENGTHS:
• High binary precision (90.5%) - accurate when confident
• Good AUC-PR (0.84) - effective ranking

✅ POST-CHAT STRENGTHS:  
• Fast response times for interactive conversations
• High faithfulness - recommendations grounded in KG data
• Flexible query understanding handles natural language

⚠️ COMBINED AREAS FOR IMPROVEMENT:
• Both show room for improvement in recall/coverage
• Query understanding could be enhanced for complex queries
• Consider GNN layers for better embeddings

📈 OVERALL PROJECT STATUS:
• Knowledge Graph: ✓ Complete
• KGNN Recommendations: ✓ Working (22% targets)
• RAG Chatbot: ✓ Working  
• Post-Chat Metrics: ✓ Evaluated

🎯 NEXT STEPS:
1. Integrate GNN layers for learned embeddings
2. Improve constraint parsing for edge cases
3. Add multi-turn conversation memory
4. Generate explanations for recommendations
""")
